# K-Nearest Neighbors (KNN) – Hotel Booking Cancellation Prediction

**Assignment model 3 of 5**

This notebook implements KNN using the same training and held-out test partitions used by Logistic Regression and Decision Tree.

- **Target:** `is_canceled` (`1` = cancelled; `0` = not cancelled)
- **Split:** common stratified 80:20 split, `random_state=42`
- **Preprocessing:** numerical median imputation and scaling; categorical `Unknown` imputation and one-hot encoding
- **Dimensionality reduction:** `TruncatedSVD` with 30 components, fitted on training data only, to make neighbor search practical after one-hot encoding. The explained-variance ratio is reported below.
- **Classifier:** distance-weighted KNN with 15 neighbors and Euclidean distance
- **Evaluation:** Accuracy, AUC, Precision, Recall, F1 Score and Matthews Correlation Coefficient (MCC)

Run `notebooks/01_data_preparation.ipynb` first. Do not create a new train-test split here. The held-out test data is used only for final evaluation.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "is_canceled"
MODEL_NAME = "K-Nearest Neighbors"
N_COMPONENTS = 30
N_NEIGHBORS = 15


## 2. Load the shared training and test partitions

In [ ]:
# Locate the project root whether this notebook is run from the root or model/.
root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (candidate.resolve() for candidate in root_candidates
     if (candidate / "data" / "train_data.csv").exists()
     and (candidate / "test_data.csv").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/train_data.csv and test_data.csv. "
        "Run notebooks/01_data_preparation.ipynb first."
    )

RESULTS_DIR = REPO_ROOT / "results"
ARTIFACTS_DIR = REPO_ROOT / "model" / "artifacts"
PLOTS_DIR = RESULTS_DIR / "visualizations"
for directory in (RESULTS_DIR, ARTIFACTS_DIR, PLOTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

train_data = pd.read_csv(REPO_ROOT / "data" / "train_data.csv")
test_data = pd.read_csv(REPO_ROOT / "test_data.csv")

if TARGET not in train_data.columns or TARGET not in test_data.columns:
    raise KeyError(f"Both data files must contain target column {TARGET!r}.")
if list(train_data.columns) != list(test_data.columns):
    raise ValueError("Training and test columns do not match or are in a different order.")
if set(train_data[TARGET].dropna().unique()) != {0, 1} or set(test_data[TARGET].dropna().unique()) != {0, 1}:
    raise ValueError("Expected binary target values 0 and 1 in both partitions.")

X_train = train_data.drop(columns=[TARGET]).copy()
y_train = train_data[TARGET].astype(int).copy()
X_test = test_data.drop(columns=[TARGET]).copy()
y_test = test_data[TARGET].astype(int).copy()

print(f"Project root: {REPO_ROOT}")
print(f"Training set: {X_train.shape[0]:,} rows × {X_train.shape[1]} predictors")
print(f"Test set:     {X_test.shape[0]:,} rows × {X_test.shape[1]} predictors")
print("Training target proportions (%):")
display(y_train.value_counts(normalize=True).sort_index().mul(100).round(2).rename(index={0: "Not cancelled (0)", 1: "Cancelled (1)"}).to_frame("percent"))
print("Test target proportions (%):")
display(y_test.value_counts(normalize=True).sort_index().mul(100).round(2).rename(index={0: "Not cancelled (0)", 1: "Cancelled (1)"}).to_frame("percent"))


## 3. Build preprocessing and KNN pipeline

KNN is distance-based, so numerical features must be scaled. Categorical values are imputed and one-hot encoded. One-hot encoding creates a relatively high-dimensional sparse matrix; `TruncatedSVD` reduces it to 30 components while retaining the main information patterns. SVD and the final component scaling are fitted **only on the training data** because they are part of the pipeline.

The SVD step is a practical modelling choice for this dataset size. It means this KNN model measures distance in the learned 30-dimensional representation rather than in the full one-hot feature space. The amount of variance retained is recorded so this modelling trade-off is transparent.

In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

if not numeric_features and not categorical_features:
    raise ValueError("No usable predictors were found.")

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Support both current and older scikit-learn OneHotEncoder argument names.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", onehot),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("svd", TruncatedSVD(n_components=N_COMPONENTS, n_iter=7, random_state=RANDOM_STATE)),
    ("component_scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights="distance",
        metric="minkowski",
        p=2,
        algorithm="kd_tree",
        leaf_size=40,
        n_jobs=-1,
    )),
])

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print("KNN configuration:")
print(model_pipeline.named_steps["classifier"])


## 4. Train the model

In [ ]:
# The complete preprocessing + SVD + KNN pipeline is fitted on training rows only.
start_time = time.time()
model_pipeline.fit(X_train, y_train)
training_seconds = time.time() - start_time

svd_model = model_pipeline.named_steps["svd"]
transformed_feature_count = model_pipeline.named_steps["classifier"]._fit_X.shape[1]
print(f"Training completed in {training_seconds:.2f} seconds.")
encoded_feature_count = model_pipeline.named_steps["preprocessor"].transform(X_train.iloc[:1]).shape[1]
print(f"Encoded feature dimensions before SVD: {encoded_feature_count}")
print(f"SVD components: {N_COMPONENTS}")
print(f"Cumulative explained-variance ratio: {svd_model.explained_variance_ratio_.sum():.4f}")
print(f"KNN fitted feature dimensions: {transformed_feature_count}")


## 5. Evaluate on the held-out test set

The six required metrics are calculated on the common held-out test set. Cancellation (`is_canceled = 1`) is the positive class. AUC uses the predicted probability for class 1. We report held-out test results only; KNN resubstitution scores on the training data can be misleading because each training point is its own nearest neighbour.

In [ ]:
start_time = time.time()
y_pred = model_pipeline.predict(X_test)
y_probability = model_pipeline.predict_proba(X_test)[:, 1]
evaluation_seconds = time.time() - start_time

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_probability),
    "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "F1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred),
}

metric_table = pd.DataFrame({"Metric": list(test_metrics.keys()), "Test": list(test_metrics.values())})
metric_table["Test"] = metric_table["Test"].astype(float).round(4)
display(metric_table)
print(f"Test evaluation completed in {evaluation_seconds:.2f} seconds.")
print("Classification report (cancelled booking = positive class):")
print(classification_report(y_test, y_pred, target_names=["Not cancelled", "Cancelled"], zero_division=0))

# Update exactly one row for this model in the shared comparison file.
comparison_path = RESULTS_DIR / "model_comparison.csv"
new_row = pd.DataFrame([{"Model": MODEL_NAME, **test_metrics}])
if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    if "Model" in existing.columns:
        existing = existing[existing["Model"] != MODEL_NAME]
        comparison = pd.concat([existing, new_row], ignore_index=True)
    else:
        comparison = new_row
else:
    comparison = new_row
comparison.to_csv(comparison_path, index=False, float_format="%.6f")

pd.DataFrame([{"Dataset": "Test", **test_metrics}]).to_csv(
    RESULTS_DIR / "knn_metrics.csv", index=False, float_format="%.6f"
)
pd.DataFrame({
    "actual_is_canceled": y_test.to_numpy(),
    "predicted_is_canceled": y_pred,
    "predicted_probability_cancelled": y_probability,
}).to_csv(RESULTS_DIR / "knn_test_predictions.csv", index=False, float_format="%.6f")

print("Updated model comparison:")
display(pd.read_csv(comparison_path).round(4))


## 6. Confusion matrix and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Not cancelled (0)", "Cancelled (1)"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("KNN – Confusion Matrix")
RocCurveDisplay.from_predictions(y_test, y_probability, ax=axes[1], name="KNN")
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random baseline")
axes[1].set_title("KNN – ROC Curve")
axes[1].legend(loc="lower right")
fig.tight_layout()
confusion_roc_path = PLOTS_DIR / "knn_confusion_matrix_roc.png"
fig.savefig(confusion_roc_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {confusion_roc_path.relative_to(REPO_ROOT)}")


## 7. Visualise the six evaluation metrics

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
metric_names = list(test_metrics.keys())
metric_values = [test_metrics[name] for name in metric_names]
bars = ax.bar(metric_names, metric_values)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("KNN – Held-out Test Metrics")
ax.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.3f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
metrics_plot_path = PLOTS_DIR / "knn_metrics.png"
fig.savefig(metrics_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {metrics_plot_path.relative_to(REPO_ROOT)}")


## 8. Save the fitted pipeline and configuration

In [ ]:
pipeline_path = ARTIFACTS_DIR / "knn_pipeline.joblib"
config_path = ARTIFACTS_DIR / "knn_config.json"
joblib.dump(model_pipeline, pipeline_path)

config = {
    "model_name": MODEL_NAME,
    "target": TARGET,
    "positive_class": 1,
    "random_state": RANDOM_STATE,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "preprocessing": {
        "numeric_imputation": "median",
        "numeric_scaling": "StandardScaler",
        "categorical_imputation": "constant: Unknown",
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "dimensionality_reduction": "TruncatedSVD",
        "svd_components": N_COMPONENTS,
        "svd_explained_variance_ratio_sum": float(svd_model.explained_variance_ratio_.sum()),
        "fit_on_training_data_only": True,
    },
    "classifier": {
        "n_neighbors": N_NEIGHBORS,
        "weights": "distance",
        "metric": "minkowski",
        "p": 2,
        "algorithm": "kd_tree",
        "leaf_size": 40,
    },
    "training_seconds": round(training_seconds, 3),
    "test_evaluation_seconds": round(evaluation_seconds, 3),
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"Saved fitted pipeline: {pipeline_path.relative_to(REPO_ROOT)}")
print(f"Saved configuration: {config_path.relative_to(REPO_ROOT)}")


## 9. Observations and limitations

Record observations from the actual held-out metrics after execution. KNN is a distance-based, non-parametric classifier. Scaling is important because otherwise large-scale numerical features could dominate distances. Distance weighting gives closer neighbours more influence than farther neighbours.

For this high-dimensional dataset, the pipeline uses 30-component TruncatedSVD after one-hot encoding. The explained-variance ratio is reported in the training output and configuration file. This reduction makes the neighbour search tractable, but it can discard some information. KNN performance should therefore be interpreted as the performance of this complete preprocessing + dimensionality-reduction + KNN pipeline, not as KNN on the unreduced raw columns.

The positive class is booking cancellation (`1`). Interpret Precision and Recall together, and compare MCC and AUC alongside Accuracy because the target is not evenly distributed. Do not select the overall winner until all five models have been run on the same held-out test set.